### Setup and Loading Data

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
import joblib

# Load dataset
df = pd.read_excel(r"E:\project_dv\paper1\mmc2.xlsx", sheet_name="Raw Data")
df_social = pd.read_csv(r"E:\project_dv\paper1\social_df.csv")

# Features & Targets
q_cols = df.columns[1:21].tolist()
X = df[q_cols].apply(lambda col: col.astype(str).str.strip().lower())  # Clean whitespace only
y_ucla = df_social["ucla_score"]
y_phq = df_social["PHQ9_score"]
y_gad = df_social["GAD7_score"]


### Define Column Groups

In [ ]:
# Binary Yes/No Columns
binary_cols = [
     q_cols[13], q_cols[14], q_cols[15], q_cols[18]
]

# Ordinal Columns with Values
ordinal_cols = {
    q_cols[4]: ["Less than 2-year", "2-5 years", "5-10 years", "More than 10 years"],
    q_cols[5]: ["Less than 1 per day", "1-2 per day", "3-5 per day", "More than 5 per day"],
    q_cols[6]: ["Less than 1 hour", "1-3 hours", "3-5 hours", "More than 5 hours"],
    q_cols[7]: ['Frequently at anytime', 'Evening', 'Night to late night', 'Afternoon', 'Morning'],
    q_cols[8]: ["Less than 500", "500-2000", "2000-4000", "More than 4000"],
    q_cols[9]: ['Most of them', 'Few of them', 'All of them', 'Many of them'],
    q_cols[10]: ["Less than 5", "5-10", "10-20", "More than 20"],
    q_cols[16]: ['Always', 'Sometimes', 'Not at all'],
    q_cols[17]: ['All the times', 'Most of the times', 'Never'],
    q_cols[19]: ['No, I am not trying', 'Trying to reduce the use', 
                 'Yes, I am trying and I have reduced using it', 
                 'Yes, I am trying but can’t', 'Not trying', 'Trying to stop the use']
}

# One-Hot Encoded Columns
onehot_cols = [
    q_cols[1], q_cols[2], q_cols[3], q_cols[11], q_cols[12]
]


In [53]:
X[q_cols[4]].unique()

array(['5-10 years', 'Less than 2-year', '2-5 years',
       'More than 10 years'], dtype=object)

### Define Transfomers and Preprocessors

In [57]:
ordinal_transformer = OrdinalEncoder(categories=[ordinal_cols[col] for col in ordinal_cols])
onehot_transformer = OneHotEncoder(handle_unknown='ignore')
binary_transformer = OrdinalEncoder(categories=[["no", "yes"]])

preprocessor = ColumnTransformer(transformers=[
    ('ord', ordinal_transformer, list(ordinal_cols.keys())),
    ('ohe', onehot_transformer, onehot_cols),
    ('bin', binary_transformer, binary_cols)
])


### Define a Reusable Pipeline Function

In [58]:
def train_pipeline(X, y, filename, model=None):
    if model is None:
        model = RandomForestRegressor(random_state=42)

    pipeline = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', model)
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    pipeline.fit(X_train, y_train)

    joblib.dump(pipeline, filename)
    print(f"Pipeline saved: {filename}")


### Train and Save Pipelines

In [59]:
train_pipeline(X, y_ucla, "ucla_pipeline.pkl")
train_pipeline(X, y_phq, "phq_pipeline.pkl")
train_pipeline(X, y_gad, "gad_pipeline.pkl")


ValueError: Shape mismatch: if categories is an array, it has to be of shape (n_features,).

In [ ]:
len(ordinal_cols.keys()) == len([ordinalcategories])


SyntaxError: invalid syntax (2571029866.py, line 1)

In [61]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import joblib

# Load data
df = pd.read_excel(r"E:\project_dv\paper1\mmc2.xlsx", sheet_name="Raw Data")
df_social = pd.read_csv(r"E:\project_dv\paper1\social_df.csv")

# Targets
y_ucla = df_social["ucla_score"]
y_phq = df_social["PHQ9_score"]
y_gad = df_social["GAD7_score"]

# Social media questions
q_cols = df.columns[1:21].tolist()
df = df[q_cols].apply(lambda col: col.astype(str).str.strip().str.lower())

# Manual Encoding Function
def manual_encode(df):
    df_encoded = pd.DataFrame()

    # Binary
    df_encoded['Q1'] = df[q_cols[0]].map({'yes': 1, 'no': 0})
    df_encoded['Q14'] = df[q_cols[13]].map({'yes': 1, 'no': 0})
    df_encoded['Q15'] = df[q_cols[14]].map({'yes': 1, 'no': 0})
    df_encoded['Q16'] = df[q_cols[15]].map({'yes': 1, 'no': 0})
    df_encoded['Q19'] = df[q_cols[18]].map({'yes': 1, 'no': 0})

    # Ordinal
    ordinal_mappings = {
        q_cols[4]: ["less than 2 year", "2-5 years", "5-10 years", "more than 10 years"],
        q_cols[5]: ["less than 1 per day", "1-2 per day", "3-5 per day", "more than 5 per day"],
        q_cols[6]: ["less than 1 hour", "1-3 hours", "3-5 hours", "more than 5 hours"],
        q_cols[7]: ["morning", "afternoon", "evening", "night"],
        q_cols[8]: ["less than 500", "500-2000", "2000-4000", "more than 4000"],
        q_cols[9]: ["few of them", "many of them", "most of them", "all of them"],
        q_cols[10]: ["less than 5", "5-10", "10-20", "more than 20"],
        q_cols[16]: ["not at all", "sometimes", "always"],
        q_cols[17]: ["never", "most of the times", "all the times"],
        q_cols[19]: ["no, i am not trying", "not trying", "trying to reduce the use",
                     "yes, i am trying but can’t", "yes, i am trying and i have reduced using it", "trying to stop the use"]
    }

    for col, categories in ordinal_mappings.items():
        df_encoded[col] = df[col].map({cat: i for i, cat in enumerate(categories)})

    # One-Hot
    onehot_cols = [q_cols[1], q_cols[2], q_cols[3], q_cols[11], q_cols[12]]
    df_encoded = pd.get_dummies(df_encoded.join(df[onehot_cols]), columns=onehot_cols, drop_first=True)

    return df_encoded

# Encode
X_encoded = manual_encode(df)

# Train models
def train_and_save_model(X, y, filename):
    model = RandomForestRegressor(random_state=42)
    X_train, X_test, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    joblib.dump(model, filename)
    print(f"Model saved: {filename}")

train_and_save_model(X_encoded, y_ucla, "ucla_model.pkl")
train_and_save_model(X_encoded, y_phq, "phq_model.pkl")
train_and_save_model(X_encoded, y_gad, "gad_model.pkl")


ValueError: Input X contains NaN.
RandomForestRegressor does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [62]:
X_encoded.head()

,Q1,Q14,Q15,Q16,Q19,5. How long have you been using a social media account?,6. How frequently do you post (upload status or add photos/videos) on social media?,7. How much time do you spend daily in social media?,8. When do you usually use social media?,9. How many friends do you have on social media?,...,13. What contents do you mainly look for in your social media news feed?_motivational/ informative contents,13. What contents do you mainly look for in your social media news feed?_nothing specific,13. What contents do you mainly look for in your social media news feed?_studying,13. What contents do you mainly look for in your social media news feed?_to buy products from various online pages/groups,"13. What contents do you mainly look for in your social media news feed?_to get updates of job circular, to learn from job related post/experience",13. What contents do you mainly look for in your social media news feed?_to stay connected with people,"13. What contents do you mainly look for in your social media news feed?_to stay connected, get updates from people and get pleasure from memes",13. What contents do you mainly look for in your social media news feed?_to talk with husband,13. What contents do you mainly look for in your social media news feed?_updates of current affairs/news,13. What contents do you mainly look for in your social media news feed?_updates of people
0,1,1,0,0,0,2.0,0,1,NaN,0,...,False,False,False,False,False,False,False,False,False,True
1,1,1,0,0,1,2.0,2,3,NaN,1,...,False,False,False,False,False,True,False,False,False,False
2,1,1,0,1,1,2.0,0,2,NaN,0,...,True,False,False,False,False,False,False,False,False,False
3,1,1,1,1,1,NaN,0,2,NaN,1,...,False,False,False,False,False,False,False,False,True,False
4,1,1,1,1,1,NaN,3,3,NaN,2,...,False,False,False,False,False,False,False,False,False,False
